# Agent 3 — Smart Market Comparison & Recommendation

**Farmer inputs (via Orchestrator):** `millet`, `grade`, `quantity`, `location_text`

**What Agent 3 does:**
1. **Geocode** location text → lat/lon using Nominatim (OpenStreetMap, free)
2. **Compare** all APMCs within range using OSRM real road distance
3. **Bulk-aware transport**: truck-rental formula scales with quantity
4. **MSP floor check** — warns farmer if price below govt MSP
5. **Break-even analysis** — tells farmer the quantity needed to profit from travel
6. **Shared transport insight** — shows savings if partnering with other farmers
7. **Human-readable insights** — ready for Streamlit display

**All APIs used:** OSRM + Nominatim. Both free OpenStreetMap. Zero signup, zero cost.

## Cell 1 — Load model, encoders, markets

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import pandas as pd
import numpy as np
import pickle, json, requests
from datetime import datetime

DRIVE = "/content/drive/MyDrive/MilletSaarthi"

with open(f"{DRIVE}/models/price_model.pkl", "rb") as f:
    price_model = pickle.load(f)
with open(f"{DRIVE}/encoders.pkl", "rb") as f:
    encoders = pickle.load(f)
with open(f"{DRIVE}/features.txt") as f:
    FEATURES = f.read().strip().split(",")
with open(f"{DRIVE}/data/apmc_markets.json") as f:
    md = json.load(f)

MARKETS = md["markets"]
TRUCK_RENTAL_PER_KM = md["truck_rental_per_km"]
TRUCK_CAPACITY = md["truck_capacity_quintals"]
MSP = md["msp_per_quintal"]

print(f"✅ {len(MARKETS)} APMCs | truck ₹{TRUCK_RENTAL_PER_KM}/km | capacity {TRUCK_CAPACITY}q")
print(f"   MSP: {MSP}")

## Cell 2 — Nominatim geocoding (location text → lat/lon)

Free OpenStreetMap geocoder. Required: custom User-Agent header.

In [ ]:
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
HEADERS = {"User-Agent": "MilletSaarthi/1.0 (academic project)"}
_geo_cache = {}

def geocode(location_text: str) -> dict:
    """Convert 'Baramati, Maharashtra' → {lat, lon, display_name, district}."""
    if location_text in _geo_cache:
        return _geo_cache[location_text]
    try:
        r = requests.get(
            NOMINATIM_URL,
            params={"q": location_text, "format": "json", "limit": 1, "addressdetails": 1, "countrycodes": "in"},
            headers=HEADERS, timeout=10
        )
        data = r.json()
        if not data:
            return {"error": f"Could not find location: {location_text}"}
        hit = data[0]
        addr = hit.get("address", {})
        result = {
            "lat": float(hit["lat"]),
            "lon": float(hit["lon"]),
            "display_name": hit.get("display_name", location_text),
            "district": addr.get("state_district") or addr.get("county") or addr.get("city") or "Unknown",
            "state": addr.get("state", "Unknown"),
        }
        _geo_cache[location_text] = result
        return result
    except Exception as e:
        return {"error": str(e)}

# Test
print(geocode("Baramati, Maharashtra"))
print(geocode("Hubli, Karnataka"))

## Cell 3 — OSRM road distance + haversine fallback

In [ ]:
OSRM_URL = "http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
_dist_cache = {}

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def get_road_distance(lat1, lon1, lat2, lon2):
    key = (round(lat1,4), round(lon1,4), round(lat2,4), round(lon2,4))
    if key in _dist_cache:
        return _dist_cache[key]
    url = OSRM_URL.format(lon1=lon1, lat1=lat1, lon2=lon2, lat2=lat2)
    try:
        r = requests.get(url, params={"overview": "false"}, timeout=10)
        if r.status_code == 200:
            d = r.json()
            if d.get("code") == "Ok" and d.get("routes"):
                route = d["routes"][0]
                result = (round(route["distance"]/1000, 1), round(route["duration"]/60, 1), "osrm")
                _dist_cache[key] = result
                return result
    except requests.RequestException:
        pass
    km = round(haversine_km(lat1, lon1, lat2, lon2) * 1.3, 1)
    result = (km, round(km/50*60, 1), "haversine_fallback")
    _dist_cache[key] = result
    return result

print("✅ Routing ready")

## Cell 4 — Price prediction helper (uses Agent 2 model)

In [ ]:
def season_of(m):
    if m in (6,7,8,9): return "kharif"
    if m in (10,11,12,1,2,3): return "rabi"
    return "summer"

def predict_price_at(millet, grade, state, district, year, month):
    try:
        row = {
            "millet_enc":   encoders["millet"].transform([millet])[0],
            "state_enc":    encoders["state"].transform([state])[0],
            "district_enc": encoders["district"].transform([district])[0],
            "grade_enc":    encoders["grade"].transform([grade])[0],
            "season_enc":   encoders["season"].transform([season_of(month)])[0],
            "year": year, "month": month,
            "month_sin": np.sin(2*np.pi*month/12),
            "month_cos": np.cos(2*np.pi*month/12),
        }
        X = pd.DataFrame([row])[FEATURES]
        return float(price_model.predict(X)[0])
    except ValueError:
        return None

def transport_cost_per_q(distance_km, quantity_q):
    """Bulk-aware: truck hire cost divided by effective load."""
    truck_total = distance_km * TRUCK_RENTAL_PER_KM
    effective_load = min(quantity_q, TRUCK_CAPACITY)
    return truck_total / max(effective_load, 1)

print("✅ Helpers ready")

## Cell 5 — Agent 3 main: smart_market_recommendation()

In [ ]:
def smart_market_recommendation(location_text: str, millet: str, grade: str,
                                quantity_quintals: float,
                                max_distance_km: float = 300,
                                year: int = None, month: int = None) -> dict:
    if year is None or month is None:
        now = datetime.now()
        year, month = now.year, now.month

    # 1. Geocode farmer location
    geo = geocode(location_text)
    if "error" in geo:
        return {"error": f"Location not found: {location_text}"}
    flat, flon = geo["lat"], geo["lon"]

    # 2. Evaluate every APMC within range
    results = []
    for m in MARKETS:
        straight = haversine_km(flat, flon, m["lat"], m["lon"])
        if straight > max_distance_km:
            continue
        dist, dur, src = get_road_distance(flat, flon, m["lat"], m["lon"])
        if dist > max_distance_km:
            continue
        apmc_price = predict_price_at(millet, grade, m["state"], m["district"], year, month)
        if apmc_price is None:
            continue
        transport = transport_cost_per_q(dist, quantity_quintals)
        commission = apmc_price * (m["commission_pct"] / 100)
        net_q = apmc_price - transport - commission
        results.append({
            "market": m["name"], "district": m["district"], "state": m["state"],
            "distance_km": dist, "duration_min": dur, "distance_source": src,
            "apmc_price": round(apmc_price, 2),
            "transport_per_q": round(transport, 2),
            "commission_per_q": round(commission, 2),
            "net_price_per_q": round(net_q, 2),
            "total_net_revenue": round(net_q * quantity_quintals, 2),
        })

    if not results:
        return {"error": "No APMC markets found within range"}

    # Top 5 NEAREST markets (by road distance)
    nearest_5 = sorted(results, key=lambda x: x["distance_km"])[:5]
    # Best among the 5 nearest (by net price)
    best = max(nearest_5, key=lambda x: x["net_price_per_q"])
    # Local = the closest APMC
    local = nearest_5[0]
    msp = MSP.get(millet, 0)

    # 3. Shared-transport collab suggestion
    # Among the 5 nearest (excluding local), find any market that beats local under full-truck economics
    shared = None
    if quantity_quintals < TRUCK_CAPACITY:
        candidates = []
        for m in nearest_5:
            if m["market"] == local["market"]:
                continue
            full_truck_transport = (m["distance_km"] * TRUCK_RENTAL_PER_KM) / TRUCK_CAPACITY
            full_net = m["apmc_price"] - full_truck_transport - m["commission_per_q"]
            gain = full_net - local["net_price_per_q"]
            if gain > 0:
                candidates.append((gain, full_net, m))
        if candidates:
            gain, full_net, target = max(candidates, key=lambda x: x[0])
            partners_needed = int(np.ceil((TRUCK_CAPACITY - quantity_quintals) / max(quantity_quintals, 1)))
            shared = {
                "target_market": target["market"],
                "distance_km": target["distance_km"],
                "full_truck_net_per_q": round(full_net, 2),
                "gain_per_q": round(gain, 2),
                "total_gain": round(gain * quantity_quintals, 2),
                "partners_suggested": partners_needed,
                "message": (
                    f"If you team up with ~{partners_needed} other farmers to fill a 100-quintal truck to "
                    f"{target['market']} ({target['distance_km']} km), net price rises to ₹{round(full_net,2)}/q "
                    f"(+₹{round(gain,2)}/q vs selling locally). Extra ₹{round(gain*quantity_quintals,2)} for your quantity. "
                    "Ask your village cooperative or eNAM portal."
                )
            }

    # 4. MSP check
    msp_check = {
        "msp": msp,
        "your_best_net": best["net_price_per_q"],
        "status": "ABOVE_MSP" if best["net_price_per_q"] >= msp else "BELOW_MSP",
        "message": (
            f"✅ Your best net price ₹{best['net_price_per_q']} is above MSP ₹{msp}. Safe to sell."
            if best["net_price_per_q"] >= msp else
            f"⚠️ Your best net price ₹{best['net_price_per_q']} is BELOW MSP ₹{msp}. "
            "Consider FCI/NAFED govt procurement instead of APMC."
        )
    }

    # 5. Insights (human-readable bullets for Streamlit UI)
    insights = []
    if best["market"] == local["market"]:
        insights.append(
            f"📍 Your nearest market ({local['market']}) is the best choice. "
            "Do NOT be misled by middlemen promising higher prices at distant markets — "
            "after transport costs, local gives the highest net return."
        )
    else:
        gain_total = (best["net_price_per_q"] - local["net_price_per_q"]) * quantity_quintals
        insights.append(
            f"🚚 Best among 5 nearest is {best['market']} ({best['distance_km']} km). "
            f"Extra ₹{round(gain_total,2)} vs selling at your nearest ({local['market']})."
        )

    insights.append(msp_check["message"])
    if shared:
        insights.append("🤝 " + shared["message"])
    else:
        insights.append("ℹ️ No shared-transport opportunity worth it among the 5 nearest markets for your current quantity.")

    return {
        "farmer_location": {
            "input": location_text,
            "lat": flat, "lon": flon,
            "resolved": geo["display_name"],
            "district": geo["district"], "state": geo["state"],
        },
        "millet": millet, "grade": grade, "quantity_quintals": quantity_quintals,
        "query_month": f"{year}-{month:02d}",
        "local_market": local,
        "best_market": best,
        "top_5_nearest": nearest_5,
        "msp_check": msp_check,
        "shared_transport_opportunity": shared,
        "insights": insights,
        "markets_evaluated": len(results),
    }

print("✅ smart_market_recommendation() ready")

## Cell 6 — Test: Farmer from Baramati, Grade A Jowar, 10 quintals

In [ ]:
import pprint
result = smart_market_recommendation(
    location_text="Baramati, Maharashtra",
    millet="jowar", grade="A", quantity_quintals=10,
    year=2026, month=5
)
pprint.pprint(result, sort_dicts=False, width=120)

## Cell 7 — Test: Farmer from Hubli, Grade B Ragi, 50 quintals

In [ ]:
result = smart_market_recommendation(
    location_text="Hubli, Karnataka",
    millet="ragi", grade="B", quantity_quintals=50,
    year=2026, month=5
)
print("🏆 BEST (among 5 nearest):", result["best_market"]["market"], "| Net ₹"+str(result["best_market"]["net_price_per_q"]))
print("📍 NEAREST:", result["local_market"]["market"], "| Net ₹"+str(result["local_market"]["net_price_per_q"]))
print("\n💡 INSIGHTS:")
for i in result["insights"]:
    print(" -", i)
print("\n📊 TOP 5 NEAREST:")
for i, m in enumerate(result["top_5_nearest"], 1):
    print(f"  {i}. {m['market']:<32} {m['distance_km']:>5}km  net ₹{m['net_price_per_q']}")